# Product & Category Trends EDA

**Data Mode Configuration**:
- Set `USE_SAMPLE = False` in cell 1 for full dataset analysis (production)
- Set `USE_SAMPLE = True` in cell 1 for quick sample testing (development)

In [ ]:
import importlib
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

# --- Configuration ---
# set USE_SAMPLE = True for quick testing, False for full dataset analysis
USE_SAMPLE = False
TOP_N = 10  # Top N departments/products to display

# Setup project path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists() and (parent / "data").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Reload the loader module
import instacart_quality.data_access as data_access_module
importlib.reload(data_access_module)

# Load order_products with product dimensions (products + aisles + departments)
# The convenience function handles all merging automatically
df = data_access_module.load_order_products_with_product_dim(use_sample=USE_SAMPLE, project_root=PROJECT_ROOT)

print(f"Loaded {len(df):,} order lines")
print(f"Data mode: {'SAMPLE' if USE_SAMPLE else 'FULL DATASET'}")
df.head()

## Suggested Questions
- Which departments dominate order volume?
- Which aisles have highest reorder tendency?
- Which products are consistently top demand?

In [ ]:
dept_counts = df["department"].value_counts().head(TOP_N).sort_values()
plt.figure(figsize=(10, 6))
sns.barplot(x=dept_counts.values, y=dept_counts.index, orient="h")
plt.title("Top Departments by Order Lines")
plt.xlabel("Order lines")
plt.ylabel("Department")
plt.tight_layout()
plt.show()

In [ ]:
product_counts = df["product_name"].value_counts().head(TOP_N).sort_values()
plt.figure(figsize=(10, 6))
sns.barplot(x=product_counts.values, y=product_counts.index, orient="h")
plt.title("Top Products by Order Lines")
plt.xlabel("Order lines")
plt.ylabel("Product")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

GLOBAL_REORDER_RATE = df["reordered"].mean()
PRIOR_STRENGTH = 40  # Bayesian smoothing strength; higher = more conservative
MIN_LINES_DEPT = 200 if USE_SAMPLE else 3000
MIN_LINES_PRODUCT = 60 if USE_SAMPLE else 1000

def build_reorder_profile(data: pd.DataFrame, group_col: str, min_lines: int, prior_strength: int):
    profile = (
        data.groupby(group_col, dropna=False)["reordered"]
        .agg(order_lines="size", reorders="sum")
        .reset_index()
    )
    profile["raw_reorder_rate"] = profile["reorders"] / profile["order_lines"]
    profile["smoothed_reorder_rate"] = (
        profile["reorders"] + prior_strength * GLOBAL_REORDER_RATE
    ) / (profile["order_lines"] + prior_strength)
    profile["lift_vs_global"] = profile["smoothed_reorder_rate"] / GLOBAL_REORDER_RATE

    # Wilson score interval for proportion uncertainty
    z = 1.96
    n = profile["order_lines"].astype(float)
    p = profile["raw_reorder_rate"].astype(float)
    denom = 1 + (z ** 2) / n
    center = p + (z ** 2) / (2 * n)
    margin = z * np.sqrt((p * (1 - p) / n) + ((z ** 2) / (4 * (n ** 2))))
    profile["ci_low"] = (center - margin) / denom
    profile["ci_high"] = (center + margin) / denom
    profile["ci_width"] = profile["ci_high"] - profile["ci_low"]

    filtered = (
        profile[profile["order_lines"] >= min_lines]
        .sort_values(["smoothed_reorder_rate", "order_lines"], ascending=[False, False])
        .reset_index(drop=True)
    )
    return filtered

dept_reorder_adv = build_reorder_profile(
    data=df, group_col="department", min_lines=MIN_LINES_DEPT, prior_strength=PRIOR_STRENGTH
)
product_reorder_adv = build_reorder_profile(
    data=df, group_col="product_name", min_lines=MIN_LINES_PRODUCT, prior_strength=PRIOR_STRENGTH
)

print(f"Global reorder rate: {GLOBAL_REORDER_RATE:.3f}")
display(
    dept_reorder_adv[[
        "department", "order_lines", "raw_reorder_rate",
        "smoothed_reorder_rate", "lift_vs_global", "ci_low", "ci_high"
    ]].head(10)
 )
display(
    product_reorder_adv[[
        "product_name", "order_lines", "raw_reorder_rate",
        "smoothed_reorder_rate", "lift_vs_global", "ci_low", "ci_high"
    ]].head(10)
)

In [ ]:
def plot_top_bottom(profile: pd.DataFrame, label_col: str, title_prefix: str, top_n: int = 12):
    top = profile.nlargest(top_n, "smoothed_reorder_rate").sort_values("smoothed_reorder_rate")
    bottom = profile.nsmallest(top_n, "smoothed_reorder_rate").sort_values("smoothed_reorder_rate")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=False)
    sns.barplot(data=top, x="smoothed_reorder_rate", y=label_col, ax=axes[0], color="#2a9d8f")
    axes[0].axvline(GLOBAL_REORDER_RATE, color="black", linestyle="--", linewidth=1)
    axes[0].set_title(f"{title_prefix}: Highest Smoothed Reorder Rate")
    axes[0].set_xlabel("Smoothed reorder rate")
    axes[0].set_ylabel(label_col)

    sns.barplot(data=bottom, x="smoothed_reorder_rate", y=label_col, ax=axes[1], color="#e76f51")
    axes[1].axvline(GLOBAL_REORDER_RATE, color="black", linestyle="--", linewidth=1)
    axes[1].set_title(f"{title_prefix}: Lowest Smoothed Reorder Rate")
    axes[1].set_xlabel("Smoothed reorder rate")
    axes[1].set_ylabel("")

    plt.tight_layout()
    plt.show()

# Department-level view (stable because of higher support)
plot_top_bottom(dept_reorder_adv, "department", "Department", top_n=min(TOP_N, len(dept_reorder_adv)))

# Product-level view: keep only high-volume segment using adaptive threshold
if len(product_reorder_adv) > 0:
    q_cut = product_reorder_adv["order_lines"].quantile(0.75)
    high_volume_products = product_reorder_adv[product_reorder_adv["order_lines"] >= q_cut].copy()

    # Ensure enough products for top/bottom comparison on small samples
    if len(high_volume_products) < min(2 * TOP_N, len(product_reorder_adv)):
        high_volume_products = product_reorder_adv.nlargest(
            min(max(2 * TOP_N, 20), len(product_reorder_adv)), "order_lines"
        ).copy()

    plot_top_bottom(
        high_volume_products,
        "product_name",
        "Product (High Volume)",
        top_n=min(TOP_N, len(high_volume_products))
    )

    # Relationship between volume and reorder tendency
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=high_volume_products.sample(min(1500, len(high_volume_products)), random_state=42),
        x="order_lines",
        y="smoothed_reorder_rate",
        alpha=0.4,
        s=40,
        edgecolor=None
    )
    plt.axhline(GLOBAL_REORDER_RATE, color="black", linestyle="--", linewidth=1, label="Global reorder rate")
    plt.xscale("log")
    plt.title("Product-level Reorder Profile (High-volume segment)")
    plt.xlabel("Order lines (log scale)")
    plt.ylabel("Smoothed reorder rate")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No product rows available after filtering.")

## Segmentation Analysis